# nb_migrate_items

Bulk-migrate Fabric **code artifacts** (Notebooks, Pipelines, Semantic Models, Reports, KQL Dashboards, Spark Job Definitions) between workspaces using the preview `exportItemDefinitions` / `importItemDefinitions` REST APIs.

**Scope**
- Moves item **definitions only**. Data artifacts (Lakehouse, Warehouse, SQL DB, KQL databases) are **not** touched.
- Non-destructive. Source workspace items remain in place. Deletion, if desired, belongs in a separate utility.
- Intended for one-shot workspace splits / cleanups. For ongoing dev→test→prod, use Git Integration or Deployment Pipelines.

**Post-run**
- Target workspace items get new GUIDs. Chain `nb_extract_guids` against the target workspace to refresh its Variable Library.

## Prerequisites

- **SPN with Contributor role on both workspaces.** Viewer is insufficient; the export API silently omits items the caller lacks write permission on.
- **Tenant setting enabled:** *Service principals can use Fabric APIs* (Admin portal → Tenant settings).
- **Key Vault** accessible from this workspace, holding the SPN client secret.
- **Cross-item GUID references** (pipeline→lakehouse, report→semantic model, Direct Lake expressions, etc.) must already be parameterised via Variable Library. If references are hardcoded, the imported copies will point at source-workspace GUIDs until you rewrite them.
- APIs are **preview** (`beta=true` required). Endpoint paths and payload shape may shift before GA.

## Imports

In [ ]:
import base64
import json
import os
import time
from typing import Any, Optional

import msal
import requests
import notebookutils

## Configuration

Edit this cell. Everything downstream reads from these variables.

In [ ]:
# --- Workspaces ----------------------------------------------------------
source_workspace_id = ""  # GUID of the workspace you are migrating FROM
target_workspace_id = ""  # GUID of the workspace you are migrating INTO

# --- Items to migrate ----------------------------------------------------
# Explicit list. Each entry: {"displayName": "...", "type": "..."}
# Types must come from supported_item_types below.
items_to_migrate: list[dict[str, str]] = [
    # {"displayName": "nb_ingest",       "type": "Notebook"},
    # {"displayName": "pl_orchestrator", "type": "DataPipeline"},
    # {"displayName": "SalesModel",      "type": "SemanticModel"},
    # {"displayName": "SalesReport",     "type": "Report"},
]

# If True, items_to_migrate is ignored and every allowlist-matching item in the
# source workspace is migrated. Use for wholesale workspace decomposition.
migrate_all_supported = False

# --- Allowlist -----------------------------------------------------------
# Bulk APIs cover code artifacts only. Data items (Lakehouse, Warehouse,
# SQLEndpoint, SQLDatabase, KQLDatabase, Dashboard) are intentionally excluded:
# either not definition-exportable or hold data that this utility will not move.
supported_item_types = [
    "Notebook",
    "DataPipeline",
    "SemanticModel",
    "Report",
    "KQLDashboard",
    "SparkJobDefinition",
]

# --- Authentication ------------------------------------------------------
key_vault_url = "https://<kv-name>.vault.azure.net/"   # trailing slash optional
spn_client_secret_name = "fabric-migration-spn-secret" # KV secret name
tenant_id = ""      # Entra tenant GUID
spn_client_id = ""  # App registration (client) ID

# --- Flow control --------------------------------------------------------
run_export = True   # phase 1: pull definitions from source
run_import = False  # phase 2: push definitions into target. Default False so
                    # you can review the export before anything lands in target.

# --- Import behaviour ----------------------------------------------------
# True  = if target already has an item with the same displayName+type, update
#         it in place. If not, create. Makes re-runs idempotent.
# False = create-only. Fails on any name collision. Safer for first run into an
#         empty target workspace.
allow_pairing_by_name = True

# --- Optional staging ----------------------------------------------------
# Write decoded definition parts to local disk for inspection / commit to Git
# between export and import. Path must be writable from this session.
stage_to_disk = False
staging_path = "/tmp/fabric_migration_staging/"

## Constants

Endpoint paths are isolated here so you can patch them without hunting through call sites if Microsoft renames during preview.

In [ ]:
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
FABRIC_TOKEN_SCOPE = "https://api.fabric.microsoft.com/.default"

# PREVIEW endpoints. `importItemDefinitions` confirmed against MS Learn CI/CD
# tutorial (https://learn.microsoft.com/en-us/fabric/cicd/tutorial-bulkapi-cicd).
# `exportItemDefinitions` inferred by symmetry. If either returns 404, verify
# against the current OpenAPI spec: https://github.com/microsoft/fabric-rest-api-specs
BULK_EXPORT_ENDPOINT = "exportItemDefinitions"
BULK_IMPORT_ENDPOINT = "importItemDefinitions"
BETA_QUERY = "?beta=true"

# LRO polling
LRO_INITIAL_RETRY_SECONDS = 5
LRO_MAX_RETRY_SECONDS = 60
LRO_HARD_TIMEOUT_SECONDS = 1800  # 30 minutes per operation

## Authentication

SPN client-credentials flow via MSAL. Token audience is `api.fabric.microsoft.com/.default` — wrong audience is the #1 cause of 401s on these calls.

In [ ]:
def get_fabric_access_token(
    tenant_id_value: str,
    client_id_value: str,
    client_secret_value: str,
    scope: str = FABRIC_TOKEN_SCOPE,
) -> str:
    """Acquire a Fabric REST bearer token via SPN client-credentials flow.

    MSAL caches the token in-process, so repeated calls within a session
    reuse the cached token until expiry (typically ~1 hour).

    Args:
        tenant_id_value: Entra tenant GUID.
        client_id_value: App registration (client) ID for the SPN.
        client_secret_value: Client secret string, resolved from Key Vault.
        scope: OAuth scope / token audience. Defaults to Fabric REST.

    Returns:
        Bearer token string with no "Bearer " prefix.

    Raises:
        RuntimeError: MSAL returned no access_token. Error payload from Entra
            is included for debugging (bad client_id, expired secret, etc.).
    """
    app = msal.ConfidentialClientApplication(
        client_id=client_id_value,
        client_credential=client_secret_value,
        authority=f"https://login.microsoftonline.com/{tenant_id_value}",
    )
    result = app.acquire_token_for_client(scopes=[scope])
    if "access_token" not in result:
        raise RuntimeError(
            f"Token acquisition failed: {result.get('error')} "
            f"- {result.get('error_description')}"
        )
    return result["access_token"]


spn_client_secret = notebookutils.credentials.getSecret(
    key_vault_url, spn_client_secret_name
)
access_token = get_fabric_access_token(
    tenant_id_value=tenant_id,
    client_id_value=spn_client_id,
    client_secret_value=spn_client_secret,
)
auth_headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}
print("Fabric token acquired.")

## Long-Running Operation helper

Bulk export and import are LROs: the initial POST returns `202 Accepted` plus a `Location` header pointing at the operation status endpoint. Poll that until `status=Succeeded`, then GET `{Location}/result` for the payload. Honours `Retry-After` when provided; exponential backoff otherwise.

In [ ]:
def poll_lro(
    initial_response: requests.Response,
    headers: dict[str, str],
    hard_timeout_seconds: int = LRO_HARD_TIMEOUT_SECONDS,
) -> dict[str, Any]:
    """Drive a Fabric LRO to terminal state and return the parsed result body.

    Behaviour:
      - If initial_response is 200, return its JSON body immediately.
      - If 202, poll the Location header until status is terminal.
      - On Succeeded, GET `{Location}/result` for the payload. If that 404s
        (status-only operation), return the status body instead.
      - On Failed/Cancelled/Undefined, raise with the error body.

    Args:
        initial_response: Response from the kick-off POST or GET.
        headers: Auth headers reused for all polling requests.
        hard_timeout_seconds: Upper bound on total polling duration.

    Returns:
        Parsed JSON body from the final response. Shape depends on the
        originating endpoint.

    Raises:
        TimeoutError: Polling exceeded hard_timeout_seconds.
        RuntimeError: Unexpected kick-off status, missing Location header,
            HTTP error during polling, or terminal Failed/Cancelled state.
    """
    if initial_response.status_code == 200:
        return initial_response.json() if initial_response.content else {}
    if initial_response.status_code != 202:
        raise RuntimeError(
            f"Unexpected kick-off status {initial_response.status_code}: "
            f"{initial_response.text}"
        )

    operation_id = initial_response.headers.get("x-ms-operation-id", "unknown")
    operation_url = initial_response.headers.get("Location")
    if not operation_url:
        raise RuntimeError(
            f"202 for operation {operation_id} but no Location header returned."
        )

    print(f"LRO started. Operation: {operation_id}")
    start_time = time.monotonic()
    retry_seconds = LRO_INITIAL_RETRY_SECONDS
    last_response = initial_response

    while True:
        if time.monotonic() - start_time > hard_timeout_seconds:
            raise TimeoutError(
                f"LRO {operation_id} exceeded {hard_timeout_seconds}s without "
                f"reaching terminal state."
            )

        retry_after = last_response.headers.get("Retry-After")
        sleep_seconds = (
            int(retry_after)
            if retry_after and retry_after.isdigit()
            else retry_seconds
        )
        time.sleep(sleep_seconds)

        status_response = requests.get(operation_url, headers=headers)
        last_response = status_response

        if status_response.status_code == 202:
            retry_seconds = min(retry_seconds * 2, LRO_MAX_RETRY_SECONDS)
            continue
        if not status_response.ok:
            raise RuntimeError(
                f"LRO {operation_id} poll failed "
                f"{status_response.status_code}: {status_response.text}"
            )

        status_body = status_response.json() if status_response.content else {}
        operation_status = status_body.get("status")

        if operation_status == "Succeeded":
            result_url = f"{operation_url}/result"
            result_response = requests.get(result_url, headers=headers)
            if result_response.status_code == 404:
                # Status-only operation, no result payload.
                return status_body
            if not result_response.ok:
                raise RuntimeError(
                    f"Operation {operation_id} Succeeded but result fetch "
                    f"failed {result_response.status_code}: {result_response.text}"
                )
            return result_response.json() if result_response.content else {}

        if operation_status in ("Failed", "Cancelled", "Undefined"):
            raise RuntimeError(
                f"Operation {operation_id} terminal status={operation_status}: "
                f"{status_body}"
            )

        # "Running" / "NotStarted" — keep polling with backoff.
        retry_seconds = min(retry_seconds * 2, LRO_MAX_RETRY_SECONDS)

## Preflight

Enumerate both workspaces with the SPN before exporting anything. Catches the two common failure modes early:
1. **Wrong token audience** → 401 on first call.
2. **SPN lacks Contributor on a workspace** → 403 or empty item list.

Read-only enumeration is not a sufficient proof of write permission on the target — only the import POST can confirm that. But it rules out the obvious misconfigurations.

In [ ]:
def list_workspace_items(
    workspace_id: str,
    headers: dict[str, str],
) -> list[dict[str, Any]]:
    """Enumerate all items in a workspace.

    Args:
        workspace_id: Workspace GUID.
        headers: Auth headers.

    Returns:
        List of item dicts. Each dict includes at minimum id, displayName,
        type, workspaceId.

    Raises:
        RuntimeError: Non-2xx response. Typical causes:
            401 - wrong token audience.
            403 - SPN lacks role on workspace, or tenant setting disabled.
            404 - workspace_id is wrong.
    """
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/items"
    response = requests.get(url, headers=headers)
    if not response.ok:
        raise RuntimeError(
            f"list items failed for {workspace_id}: "
            f"{response.status_code} {response.text}"
        )
    return response.json().get("value", [])


def preflight(
    source_ws_id: str,
    target_ws_id: str,
    headers: dict[str, str],
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Verify SPN visibility on both workspaces.

    Args:
        source_ws_id: Export source workspace GUID.
        target_ws_id: Import target workspace GUID.
        headers: Auth headers.

    Returns:
        Tuple (source_items, target_items).
    """
    print(f"Preflight source: {source_ws_id}")
    source_items_list = list_workspace_items(source_ws_id, headers)
    print(f"  {len(source_items_list)} items visible to SPN")

    print(f"Preflight target: {target_ws_id}")
    target_items_list = list_workspace_items(target_ws_id, headers)
    print(f"  {len(target_items_list)} items visible to SPN")

    return source_items_list, target_items_list


source_items, target_items = preflight(
    source_ws_id=source_workspace_id,
    target_ws_id=target_workspace_id,
    headers=auth_headers,
)

## Resolve items to migrate

Turn the `items_to_migrate` config list (or `migrate_all_supported`) into concrete item records with GUIDs. Requested items not found in the source workspace raise — silent drops would make a partial migration look successful.

In [ ]:
def resolve_items_to_migrate(
    requested: list[dict[str, str]],
    source_items_list: list[dict[str, Any]],
    allowlist: list[str],
    migrate_all: bool,
) -> list[dict[str, Any]]:
    """Resolve requested items to full source-workspace item records.

    Rules:
      - If migrate_all is True, `requested` is ignored; every allowlist-matching
        item in source_items_list is returned.
      - Otherwise, each entry in `requested` is matched on (displayName, type).
      - Types outside the allowlist are skipped with a warning. This catches
        attempts to migrate Lakehouses or other data items via config error.
      - Missing items raise ValueError.

    Args:
        requested: User config from items_to_migrate.
        source_items_list: Items enumerated from the source workspace.
        allowlist: Item types eligible for bulk export/import.
        migrate_all: If True, take every allowlist-matching source item.

    Returns:
        List of source-workspace item records.

    Raises:
        ValueError: A requested item was not found in the source workspace.
    """
    source_by_key = {
        (item["displayName"], item["type"]): item for item in source_items_list
    }

    if migrate_all:
        resolved = [item for item in source_items_list if item["type"] in allowlist]
        skipped = [item for item in source_items_list if item["type"] not in allowlist]
        for item in skipped:
            print(f"  SKIP (type not in allowlist): "
                  f"{item['displayName']} [{item['type']}]")
        return resolved

    resolved = []
    for request in requested:
        display_name = request["displayName"]
        item_type = request["type"]

        if item_type not in allowlist:
            print(f"  SKIP (type not in allowlist): "
                  f"{display_name} [{item_type}]")
            continue

        key = (display_name, item_type)
        if key not in source_by_key:
            raise ValueError(
                f"Requested item not found in source workspace: "
                f"{display_name} [{item_type}]"
            )
        resolved.append(source_by_key[key])

    return resolved


items_resolved = resolve_items_to_migrate(
    requested=items_to_migrate,
    source_items_list=source_items,
    allowlist=supported_item_types,
    migrate_all=migrate_all_supported,
)

print(f"\n{len(items_resolved)} item(s) resolved for migration:")
for item in items_resolved:
    print(f"  {item['displayName']:<40} {item['type']:<20} {item['id']}")

## Bulk export

POSTs to `exportItemDefinitions` with the resolved item list, polls the LRO, and returns the full response (Base64-encoded parts keyed by item).

> **Response shape caveat.** The preview response schema is not formally published in the sources I verified. `build_import_payload` below assumes the shape `{"items": [{"id", "displayName", "type", "definition": {"parts": [...]}}]}`. If the live response keys differ, inspect `exported_definitions` and adjust `build_import_payload` accordingly.

In [ ]:
def bulk_export_item_definitions(
    workspace_id: str,
    items_list: list[dict[str, Any]],
    headers: dict[str, str],
) -> dict[str, Any]:
    """Export multiple item definitions from a source workspace in one LRO.

    PREVIEW API (beta=true). Verify the request/response schema if the API
    shape has changed since this notebook was written.

    Args:
        workspace_id: Source workspace GUID.
        items_list: Resolved items. Each needs `id` and `type`.
        headers: Auth headers.

    Returns:
        Parsed JSON body from the completed LRO.

    Raises:
        RuntimeError: Non-2xx kick-off or LRO failure.
    """
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/{BULK_EXPORT_ENDPOINT}{BETA_QUERY}"
    body = {
        "items": [{"id": item["id"], "type": item["type"]} for item in items_list],
    }
    print(f"POST {url}")
    print(f"  exporting {len(items_list)} item(s)...")

    response = requests.post(url, headers=headers, json=body)
    result = poll_lro(response, headers)

    exported_count = len(result.get("items", []))
    print(f"  export complete. items returned: {exported_count}")
    if exported_count != len(items_list):
        print(f"  WARNING: requested {len(items_list)} items but API returned "
              f"{exported_count}. Inspect the response before importing.")
    return result


exported_definitions: dict[str, Any] = {}
if run_export:
    exported_definitions = bulk_export_item_definitions(
        workspace_id=source_workspace_id,
        items_list=items_resolved,
        headers=auth_headers,
    )
else:
    print("run_export=False - skipping export phase.")

## Optional: stage export to disk

Set `stage_to_disk=True` in config to dump decoded definition parts under `staging_path` in the canonical Fabric layout (`<displayName>.<Type>/<part_path>`). Useful for eyeballing what's about to land in the target, diffing against a Git checkout, or manually editing before import.

In [ ]:
def stage_export_to_disk(
    export_result: dict[str, Any],
    output_path: str,
) -> None:
    """Write exported definition parts to disk in Fabric export layout.

    Folder layout:
        <output_path>/<displayName>.<Type>/<part_path>

    Part payloads are Base64-decoded so files are directly readable
    (notebook-content.py, definition/model.tmdl, pipeline-content.json, etc.).

    Args:
        export_result: Parsed body from bulk_export_item_definitions.
        output_path: Base directory. Created if missing.
    """
    items = export_result.get("items", [])
    os.makedirs(output_path, exist_ok=True)
    print(f"Writing {len(items)} item(s) to {output_path}")

    for item in items:
        display_name = item["displayName"]
        item_type = item["type"]
        item_root = os.path.join(output_path, f"{display_name}.{item_type}")
        os.makedirs(item_root, exist_ok=True)

        parts = item.get("definition", {}).get("parts", [])
        for part in parts:
            part_path = part["path"].lstrip("/")
            full_path = os.path.join(item_root, part_path)
            os.makedirs(os.path.dirname(full_path) or item_root, exist_ok=True)
            decoded_bytes = base64.b64decode(part["payload"])
            with open(full_path, "wb") as file_handle:
                file_handle.write(decoded_bytes)
        print(f"  {display_name}.{item_type} ({len(parts)} parts)")

    print("Staging complete.")


if run_export and stage_to_disk:
    stage_export_to_disk(exported_definitions, staging_path)
else:
    print("Staging skipped (run_export=False or stage_to_disk=False).")

## Build import payload

Transforms the per-item export response into the flat `definitionParts` array the import API expects. Each part path is prefixed with `/<displayName>.<Type>/...` — the first path segment is how the bulk import resolves which part belongs to which target item.

`.platform` files present in the export are retained; the bulk import API uses them to resolve item type and display name during create. (This differs from the single-item `updateDefinition` API, where `.platform` is forbidden.)

In [ ]:
def build_import_payload(
    export_result: dict[str, Any],
    allow_pairing: bool,
) -> dict[str, Any]:
    """Transform bulk-export response into a bulk-import request body.

    Args:
        export_result: Parsed response from bulk_export_item_definitions.
        allow_pairing: Value for options.allowPairingByName on the import.
            True  = update existing target items matched by displayName+type;
                    create absent ones. Idempotent re-runs.
            False = create-only. Any name collision fails the operation.

    Returns:
        Request body ready to POST to the bulk import endpoint.
    """
    parts_flat = []
    for item in export_result.get("items", []):
        display_name = item["displayName"]
        item_type = item["type"]
        root_prefix = f"/{display_name}.{item_type}"

        for part in item.get("definition", {}).get("parts", []):
            part_path = part["path"]
            if not part_path.startswith("/"):
                part_path = f"/{part_path}"
            parts_flat.append({
                "path": f"{root_prefix}{part_path}",
                "payload": part["payload"],
                "payloadType": part.get("payloadType", "InlineBase64"),
            })

    return {
        "definitionParts": parts_flat,
        "options": {"allowPairingByName": allow_pairing},
    }

## Bulk import

Guarded by `run_import`. If `run_export=False` in the same session and no staged export is loaded, this raises — there's nothing to import.

The API resolves item dependencies internally (e.g. semantic model before report). You do not need to order the parts.

In [ ]:
def bulk_import_item_definitions(
    workspace_id: str,
    import_body: dict[str, Any],
    headers: dict[str, str],
) -> dict[str, Any]:
    """Import item definitions in bulk into a target workspace.

    PREVIEW API (beta=true).

    Args:
        workspace_id: Target workspace GUID.
        import_body: Output of build_import_payload.
        headers: Auth headers.

    Returns:
        Parsed JSON body from the completed LRO. Typically includes an `items`
        array with the target-workspace item records (new GUIDs, displayName,
        type) for each created/updated item.

    Raises:
        RuntimeError: Non-2xx kick-off or LRO failure. Common failure causes
            documented by Microsoft include ItemsHaveProtectedLabels, naming
            conflicts when allowPairingByName=False, and unsupported item
            types in the payload.
    """
    url = f"{FABRIC_API_BASE}/workspaces/{workspace_id}/{BULK_IMPORT_ENDPOINT}{BETA_QUERY}"
    part_count = len(import_body.get("definitionParts", []))
    print(f"POST {url}")
    print(f"  importing {part_count} parts...")

    response = requests.post(url, headers=headers, json=import_body)
    result = poll_lro(response, headers)
    print(f"  import complete.")
    return result


import_result: dict[str, Any] = {}
if run_import:
    if not exported_definitions.get("items"):
        raise RuntimeError(
            "run_import=True but exported_definitions is empty. Either run "
            "with run_export=True in the same session, or load a previously "
            "staged export into exported_definitions before this cell."
        )
    import_payload = build_import_payload(
        export_result=exported_definitions,
        allow_pairing=allow_pairing_by_name,
    )
    import_result = bulk_import_item_definitions(
        workspace_id=target_workspace_id,
        import_body=import_payload,
        headers=auth_headers,
    )
else:
    print("run_import=False - skipping import. Review the export, then re-run "
          "with run_import=True.")

## Summary

Run-level recap plus pointer to the post-migration cleanup steps.

In [ ]:
print("=" * 72)
print("Migration summary")
print("=" * 72)
print(f"Source workspace:   {source_workspace_id}")
print(f"Target workspace:   {target_workspace_id}")
print(f"Items resolved:     {len(items_resolved)}")
print(f"Export phase:       {'ran' if run_export else 'skipped'}")
if run_export:
    print(f"  items exported:   {len(exported_definitions.get('items', []))}")
print(f"Import phase:       {'ran' if run_import else 'skipped'}")
if run_import and import_result:
    imported_items = import_result.get("items", [])
    print(f"  items imported:   {len(imported_items)}")
    if imported_items:
        print("\nTarget-workspace item GUIDs (new):")
        for item in imported_items:
            print(f"  {item.get('displayName', '?'):<40} "
                  f"{item.get('type', '?'):<20} "
                  f"{item.get('id', '?')}")

print("\nNext steps:")
print("  1. Run nb_extract_guids against the target workspace to refresh its")
print("     Variable Library with the new item GUIDs.")
print("  2. Open one of each item type in the portal to confirm it loads and")
print("     its Variable Library references resolve.")
print("  3. Source workspace items are untouched. Deletion is intentionally")
print("     out of scope for this utility - use a separate delete notebook")
print("     only after the target is verified.")